# Logistic regression to additive points

These cells were executed against the real project outputs. The complete training
and SQL implementation is in the linked source modules. This walkthrough reads and
verifies executed artifacts rather than silently retraining the expensive models.
Home Credit is a consumer-lending proxy; validation is random holdout, never true OOT.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
ROOT=Path.cwd()
if not (ROOT/'reports/metrics.json').exists():
    ROOT=ROOT.parent
metrics=json.loads((ROOT/'reports/metrics.json').read_text())


## Model and points

In [2]:
print(json.dumps(metrics['scorecard_design'],indent=2))
display(pd.read_csv(ROOT/'reports/vif.csv'))
display(pd.read_csv(ROOT/'reports/scorecard_points.csv').head(30))

{
  "selected_features": [
    "EXT_SOURCE_3",
    "EXT_SOURCE_2",
    "EXT_SOURCE_1",
    "bureau_avg_credit_day",
    "DAYS_EMPLOYED",
    "bureau_debt_credit_ratio",
    "OCCUPATION_TYPE",
    "age_years",
    "credit_goods_ratio",
    "previous_credit_application_ratio",
    "card_avg_utilization",
    "previous_refusal_share",
    "installment_late_share",
    "ORGANIZATION_TYPE",
    "previous_approval_share",
    "installment_recent_late_share",
    "AMT_GOODS_PRICE",
    "REGION_RATING_CLIENT_W_CITY",
    "NAME_EDUCATION_TYPE",
    "bureau_active_count"
  ],
  "selected_feature_count": 20,
  "eligible_features": 167,
  "factor": 28.85390081777927,
  "offset": 487.1228762045055,
  "base_score": 600,
  "pdo": 20,
  "base_good_bad_odds": 50,
  "intercept": -2.436673105152441,
  "coefficient": {
    "EXT_SOURCE_3": -0.6596982758394085,
    "EXT_SOURCE_2": -0.7438315794708589,
    "EXT_SOURCE_1": -0.5412638685143908,
    "bureau_avg_credit_day": -0.08519069198837201,
    "DAYS_EMPLO

,feature,vif
0,EXT_SOURCE_3,1.406768
1,EXT_SOURCE_2,1.178195
2,EXT_SOURCE_1,1.130843
3,bureau_avg_credit_day,1.421200
4,DAYS_EMPLOYED,1.696419
5,bureau_debt_credit_ratio,1.337331
6,OCCUPATION_TYPE,1.337921
7,age_years,1.759884
8,credit_goods_ratio,1.043103
9,previous_credit_application_ratio,1.096463


,bin,count,bad,good,good_dist,bad_dist,woe,iv_component,bad_rate,feature,bin_definition,coefficient,points
0,0,75751,7141,68610,0.505640,0.599195,-0.169762,0.015882,0.094269,AMT_GOODS_PRICE,"(-inf, 450000]",-0.095957,27.401493
1,1,33243,2619,30624,0.225694,0.219784,0.026532,0.000157,0.078784,AMT_GOODS_PRICE,"(450000, 675000]",-0.095957,27.944981
2,2,23740,1437,22303,0.164370,0.120611,0.309553,0.013546,0.060531,AMT_GOODS_PRICE,"(675000, 1089000]",-0.095957,28.728594
3,3,14727,711,14016,0.103298,0.059697,0.548330,0.023907,0.048279,AMT_GOODS_PRICE,"(1089000, inf]",-0.095957,29.389705
4,__MISSING__,143,8,135,0.000999,0.000713,0.336621,0.000096,0.055944,AMT_GOODS_PRICE,__MISSING__,-0.095957,28.803536
5,Higher education,36177,1937,34240,0.252343,0.162562,0.439726,0.039479,0.053542,NAME_EDUCATION_TYPE,Higher education,-0.495061,34.152763
6,Incomplete higher,4949,429,4520,0.033315,0.036036,-0.078529,0.000214,0.086684,NAME_EDUCATION_TYPE,Incomplete higher,-0.495061,26.749782
7,Lower secondary,1811,205,1606,0.011839,0.017242,-0.375918,0.002031,0.113197,NAME_EDUCATION_TYPE,Lower secondary,-0.495061,22.501744
8,Secondary / secondary special,104586,9344,95242,0.701910,0.784033,-0.110646,0.009087,0.089343,NAME_EDUCATION_TYPE,Secondary / secondary special,-0.495061,26.290997
9,__OTHER__,81,1,80,0.000593,0.000126,1.550507,0.000725,0.012346,NAME_EDUCATION_TYPE,__OTHER__,-0.495061,50.019665


## Points formula

For factor = PDO / ln(2), offset = base score - factor x ln(base good/bad odds),
score = offset - factor x logistic log odds of TARGET=1. Baseline constants are shared
across selected variables in the points table. Higher score means lower risk. Production
rounding needs separate boundary tests. Model additivity is numerically verified.